In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
import pandas as pd
no2_lagged_df = pd.read_csv("./data/no2_lagged.csv")

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# Step 1: prepare the lagged dataframe for modeling
no2_lagged_df = no2_lagged_df.copy()
no2_lagged_df["timestamp"] = pd.to_datetime(no2_lagged_df["timestamp"])
no2_lagged_df = no2_lagged_df.sort_values(["timestamp", "latitude", "longitude"]).reset_index(drop=True)

# Step 2: define the features and the target
feature_cols = ["latitude", "longitude", "elevation", "TEMP"] + [f"no2_day{i}" for i in range(1,8)]
target_col = "no2_day0"

X = no2_lagged_df[feature_cols]
y = no2_lagged_df[target_col]

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print("Features used:", feature_cols)

# Step 3: split chronologically to avoid leaking future information into training
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,
)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

# Step 4: train an XGBoost regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.6,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=32746,
    n_jobs=-1,
    tree_method="exact",
)


In [ ]:
eval_set = [(X_train, y_train), (X_test, y_test)]
xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

# 2. Extract and Plot
results = xgb_model.evals_result()
epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

In [ ]:
y_pred = xgb_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R^2: {r2:.4f}")

# Step 6: inspect which features the model relied on most
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": xgb_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_df